<a href="https://colab.research.google.com/github/Tahvia127/BUSN-20800-Final-Project-/blob/Vadim/notebooks/wdi_controls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [4]:
import zipfile

In [5]:
with zipfile.ZipFile('WDICSV.csv.zip') as z:
    with z.open('WDICSV.csv') as f:
        wdi = pd.read_csv(f)

print(wdi.shape)
print(wdi.columns.tolist()[:10])
print(wdi.head(2))

(395276, 70)
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965']
                  Country Name Country Code  \
0  Africa Eastern and Southern          AFE   
1  Africa Eastern and Southern          AFE   

                                      Indicator Name     Indicator Code  1960  \
0  Access to clean fuels and technologies for coo...     EG.CFT.ACCS.ZS   NaN   
1  Access to clean fuels and technologies for coo...  EG.CFT.ACCS.RU.ZS   NaN   

   1961  1962  1963  1964  1965  ...       2016       2017       2018  \
0   NaN   NaN   NaN   NaN   NaN  ...  18.685118  19.205632  19.742772   
1   NaN   NaN   NaN   NaN   NaN  ...   7.606712   7.926604   8.309896   

        2019      2020       2021       2022       2023  2024  2025  
0  20.332679  20.86280  21.419621  21.996456  22.541440   NaN   NaN  
1   8.704591   9.10664   9.480804   9.903209  10.288154   NaN   NaN  

[2 rows x 70 columns]


In [6]:
indicator_map = {
    'NY.GDP.MKTP.KD.ZG': 'gdp_growth',
    'DT.ODA.ALLD.CD':    'oda_received',
    'GC.XPN.TOTL.GD.ZS': 'gov_expenditure_pct_gdp',
    'SP.DYN.LE00.IN':    'life_expectancy',
    'SE.SEC.ENRR':       'school_enrol_secondary',
    'EN.ATM.CO2E.PC':    'co2_per_capita',
    'SI.POV.GINI':       'gini',
    'BX.KLT.DINV.CD.WD': 'fdi_inflows',
}

wdi_filtered = wdi[wdi['Indicator Code'].isin(indicator_map.keys())].copy()
wdi_filtered['indicator_name'] = wdi_filtered['Indicator Code'].map(indicator_map)

print('Rows after filter:', len(wdi_filtered))
print(wdi_filtered['indicator_name'].value_counts())

Rows after filter: 1862
indicator_name
gov_expenditure_pct_gdp    266
fdi_inflows                266
gdp_growth                 266
gini                       266
life_expectancy            266
oda_received               266
school_enrol_secondary     266
Name: count, dtype: int64


In [7]:
year_cols = [str(y) for y in range(2000, 2025)]

long = wdi_filtered.melt(
    id_vars=['Country Name', 'Country Code', 'indicator_name'],
    value_vars=year_cols,
    var_name='year',
    value_name='value'
)

long['year'] = long['year'].astype(int)
long = long.rename(columns={'Country Name': 'country', 'Country Code': 'countrycode'})

In [8]:
wide = long.pivot_table(index=['country', 'countrycode', 'year'],
                        columns='indicator_name', values='value').reset_index()
wide.columns.name = None

print(wide.shape)
print(wide.head(2))

(6625, 10)
       country countrycode  year  fdi_inflows  gdp_growth  gini  \
0  Afghanistan         AFG  2000     170000.0         NaN   NaN   
1  Afghanistan         AFG  2001     680000.0   -9.431974   NaN   

   gov_expenditure_pct_gdp  life_expectancy  oda_received  \
0                      NaN           55.005  1.360100e+08   
1                      NaN           55.511  4.103600e+08   

   school_enrol_secondary  
0                     NaN  
1                14.04041  


In [10]:
fill_cols = [col for col in indicator_map.values() if col in wide.columns]
print('Using columns:', fill_cols)

wide = wide.sort_values(['country', 'year'])
wide[fill_cols] = wide.groupby('country')[fill_cols].transform(lambda g: g.ffill())

Using columns: ['gdp_growth', 'oda_received', 'gov_expenditure_pct_gdp', 'life_expectancy', 'school_enrol_secondary', 'gini', 'fdi_inflows']


In [11]:
for col in fill_cols:
    pct_null = wide[col].isna().mean()
    print(f'{col}: {pct_null:.1%} null')

gdp_growth: 2.7% null
oda_received: 22.1% null
gov_expenditure_pct_gdp: 39.7% null
life_expectancy: 0.0% null
school_enrol_secondary: 12.2% null
gini: 44.8% null
fdi_inflows: 7.4% null


In [12]:
import os
os.makedirs('data/interim', exist_ok=True)
wide.to_parquet('data/interim/wdi_controls.parquet', index=False)
print('\nSaved wdi_controls.parquet:', wide.shape)


Saved wdi_controls.parquet: (6625, 10)


In [14]:
# Note: co2_per_capita (EN.ATM.CO2E.PC) was not found in WDICSV.csv.zip
# so fill_cols filters to only existing columns rather than using all 8 indicators